# Load positive and negative interactions

We will compute additional embeddings for another 200 mice interactions. New complete data provided by Lucía.

Excluding previously computed ones in experiment4.

In [1]:
!pip install biopython

In [2]:
import numpy as np
from Bio import SeqIO
from pathlib import Path

# Conexión a Google Drive
#from google.colab import drive
#drive.mount('/content/drive')

In [3]:
import os

# To find out the absolute path of the working directory:
# os.getcwd()

os.listdir("/home/jovyan/TFG/")


['distribucion_prot_pos_neg_train_nivel3.png',
 'create_batch.sh',
 'Repetitiion_percentage_CV.ipynb',
 'mannwhitneyu_prot_ef.png',
 'go-basic.obo',
 'distribucion_grupos_prot_seq_pos_neg_train_nivel2.png',
 'Interacciones_Entrenamiento_CV_estricto.csv',
 'inputs_RepCientifica_mmseqs_batches',
 'run_Efectores_alphafold.log',
 'classify.ipynb',
 'pca_ef.png',
 'alphafold3_databases_2',
 'run_solo_Efectores_mmseqs.log',
 'distribucion_ef_pos_neg_train_nivelc3_filtrado.png',
 'distribucion_grupos_ef_pos_neg_train_nivel1_5.png',
 'mmseqs_results_similarity_30',
 'run_Efectores_alphafold_all.log',
 'distribucion_prot_pos_neg_train_nivel1_5.png',
 'assign_labels_Efectores_no_redundancies.ipynb',
 'tsne_ef.png',
 'distribucion_prot_pos_neg_train_nivelc3.png',
 'input_solo_Efectores_mmseqs',
 'resultados_CV',
 'group_effectors_by_function.ipynb',
 'tsne_random_prot_ef.png',
 'predicciones_efector_proteina_dudosos.csv',
 'distribucion_prot_pos_train.png',
 'output_RepCientifica_alphafold_batche

In [4]:
import pandas as pd

path = "/home/jovyan/TFG/"

#positive_df = pd.read_csv(path + "Interacciones_Positivas_Raton_Completo.csv")
#negative_df = pd.read_csv(path + "Interacciones_Negativas_Raton_Completo.csv")
#positive_df.shape, negative_df.shape

effector_protein_df = pd.read_csv(path + "Interacciones_EffectorProteina_LiteratureOnly_Ordenadas_NleG.csv")
effector_protein_df.shape

(5472, 6)

Check that there are no duplicate interactions:

In [5]:
#positive_df.drop_duplicates(subset=["Head", "Tail"], keep=False).shape

In [6]:
#negative_df.drop_duplicates(subset=["Head", "Tail"], keep="first").shape

There are duplicates in the negative set. We will leave the negative interaction with less evidence.

In [7]:
#negative_df = negative_df.drop_duplicates(subset=["Head", "Tail"], keep="last")

In [8]:
#positive_df.head()

In [9]:
#negative_df.head()

effector_protein_df.head()

,Effector,Protein,ProteinGeneName,Shared_Connections,Shortest_Path,Is_Connected
0,EspL,O89110,Casp8,4,1.0,True
1,EspL,Q60855,Ripk1,3,1.0,True
2,NleB,O89110,Casp8,2,1.0,True
3,NleA,Q8R4B8,Nlrp3,1,1.0,True
4,NleA,Q9D8T2,Gsdmd,1,1.0,True


# Check node degree

In [10]:
# No necesitamos comprobar los nodos porque ya estamos con el conjunto de aplicación, no vamos a entrenar con datos pos y neg

# def get_node_degrees(pos_sel, neg_sel):
#     prots_pos, degree_pos = np.unique(np.concatenate([pos_sel["Head"].values, pos_sel["Tail"].values]), return_counts=True)
#     prots_neg, degree_neg = np.unique(np.concatenate([neg_sel["Head"].values, neg_sel["Tail"].values]), return_counts=True)

#     node_degrees = pd.DataFrame({
#         "protein": np.union1d(prots_pos, prots_neg)
#     })

#     node_degrees["pos_degree"] = node_degrees["protein"].map(dict(zip(prots_pos, degree_pos))).fillna(0).astype(int)
#     node_degrees["neg_degree"] = node_degrees["protein"].map(dict(zip(prots_neg, degree_neg))).fillna(0).astype(int)
#     node_degrees["abs_degree_dif"] = (node_degrees["pos_degree"] - node_degrees["neg_degree"]).abs()
#     return node_degrees

## Disjoint train/validation/test sets

Since any overlap in the unique proteins that compose the training, validation and test sets, could mean learning by node degree alone, we will make perfectly disjoint groups. Sequence similarity will not be taken into account since access to readily available similarity matrices was not possible and calculating all pairwise similarities is unfeasable.

Calculate all unique proteins:

In [11]:
# uniq_prots = np.unique(np.concatenate((positive_df[["Head", "Tail"]].values.flatten(), negative_df[["Head", Tail"]].values.flatten())))
uniq_prots = np.unique(effector_protein_df["Protein"])
uniq_prots.shape

(171,)

Calculate all unique effectors:

In [12]:
uniq_eff = np.unique(effector_protein_df["Effector"])
uniq_eff.shape

(32,)

In [13]:
uniq_eff

array(['EscF', 'EspF', 'EspG', 'EspH', 'EspJ', 'EspK', 'EspL', 'EspM',
       'EspN', 'EspO', 'EspS', 'EspT', 'EspV', 'EspZ', 'LPS', 'Map',
       'NleA', 'NleB', 'NleC', 'NleD', 'NleE', 'NleF', 'NleG1', 'NleG7',
       'NleG8', 'NleH', 'NleK', 'NleL', 'NleN', 'NleO', 'Tir', 'pORF8'],
      dtype=object)

Get the sequences from proteins and effectors in fasta format: 

In [14]:
prots_seq_dict = {}

for protein in uniq_prots:
    try:
        records = list(SeqIO.parse(f"{path}Secuencias_ProteinaProteina/{protein}.fasta", format="fasta-pearson"))
        if len(records) == 0:
            print(f"Skipping {protein}: No sequence found in file.")
            continue
        if len(records) > 1:
            print(f"{protein} has more than one sequence!")
        prots_seq_dict[protein] = str(records[0].seq)

    except Exception as e:
        print(f"Fail at {protein}")
        print(records)
        raise e

In [15]:
# Check if they are being correctly stored
# dict(list(prots_seq_dict.items())[:2])

In [16]:
# Repeat for effectors

eff_seq_dict = {}

for effector in uniq_eff:
    try:
        records = list(SeqIO.parse(f"{path}Secuencias_Efectores/{effector}_ICC168.fasta", format="fasta-pearson"))
        if len(records) == 0:
            print(f"Skipping {effector}: No sequence found in file.")
            continue
        if len(records) > 1:
            print(f"{effector} has more than one sequence!")
        eff_seq_dict[effector] = str(records[0].seq)

    except Exception as e:
        print(f"Fail at {effector}")
        print(records)
        raise e

In [59]:
# There is no problematic file

# Inspect the problematic file
# try:
#     with open(f"{path}Secuencias_ProteinaProteina/E9Q0A3.fasta", "r") as f:
#         file_content = f.read()
#         print(f"Content of E9Q0A3.fasta:\n{file_content}")
# except FileNotFoundError:
#     print("File E9Q0A3.fasta not found.")
# except Exception as e:
#     print(f"Error reading file E9Q0A3.fasta: {e}")

In [18]:
# # We are not going to filter sequences by their length
# len_dict = {k: len(v) for k, v in seq_dict.items()}
# positive_df.loc[:, "Head_len"] = positive_df.loc[:, "Head"].map(len_dict)
# positive_df.loc[:, "Tail_len"] = positive_df.loc[:, "Tail"].map(len_dict)
# negative_df.loc[:, "Head_len"] = negative_df.loc[:, "Head"].map(len_dict)
# negative_df.loc[:, "Tail_len"] = negative_df.loc[:, "Tail"].map(len_dict)

In [19]:
# THRESHOLD = 1536 # We use a very small bucket to make AlphaFold3 predictions faster
# mask_short_positive = positive_df.apply(lambda x: x["Head_len"] + x["Tail_len"], axis=1) < THRESHOLD
# mask_short_negative = negative_df.apply(lambda x: x["Head_len"] + x["Tail_len"], axis=1) < THRESHOLD
# positive_short = positive_df[mask_short_positive]
# negative_short = negative_df[mask_short_negative]
# print(len(positive_short), len(negative_short))
# print(len(positive_short)/len(positive_df), "remaining of positives")
# print(len(negative_short)/len(negative_df), "remaining of negatives")

Partition randomly in three blocks:

In [20]:
# uniq_prots = np.unique(np.concatenate((positive_short[["Head", "Tail"]].values.flatten(), negative_short[["Head", "Tail"]].values.flatten())))
# uniq_prots.shape

In [21]:
# np.random.seed(0)
# np.random.shuffle(uniq_prots)
# n = uniq_prots.shape[0]
# P0 = uniq_prots[:int(0.6*n)]
# P1 = uniq_prots[int(0.6*n):int(0.8*n)]
# P2 = uniq_prots[int(0.8*n):]

In [22]:
# filter_byblock = lambda df, block: df[df["Head"].isin(block) & df["Tail"].isin(block)]

In [23]:
# pos_P0 = filter_byblock(positive_short, P0)
# neg_P0 = filter_byblock(negative_short, P0)
# pos_P0.shape[0], neg_P0.shape[0]

In [24]:
# pos_P1 = filter_byblock(positive_short, P1)
# neg_P1 = filter_byblock(negative_short, P1)
# pos_P1.shape[0], neg_P1.shape[0]

In [25]:
# pos_P2 = filter_byblock(positive_short, P2)
# neg_P2 = filter_byblock(negative_short, P2)
# pos_P2.shape[0], neg_P2.shape[0]

Show distribution of positive and negative degrees:

In [26]:
# get_protein_counts = lambda df: np.unique(df[["Head", "Tail"]], return_counts=True)[1]

In [27]:
# import seaborn as sns
# import matplotlib.pyplot as plt

# def plot_degrees(posdf, negdf, xlim=None):
#     pos_degree, neg_degree = (get_protein_counts(df) for df in (posdf, negdf))
#     fig, ax = plt.subplots(figsize=(5,4))
#     sns.histplot(data=pos_degree, discrete=True, stat="density", alpha=0.5, color="blue", label="Positive degree", ax=ax)
#     sns.histplot(data=neg_degree, discrete=True, stat="density", alpha=0.5, color="orange", label="Negative degree", ax=ax)
#     plt.xlim(xlim)
#     plt.legend()
#     plt.show()

In [28]:
# plot_degrees(pos_P0, neg_P0)

In [29]:
# plot_degrees(pos_P1, neg_P1)

In [30]:
# plot_degrees(pos_P2, neg_P2)

Negative degrees follow a very different distribution than positive degrees. In principle, that is bad, but I'm not sure the model could use it in any way to perform better in validation, since proteins are not shared.

Balance the dataset by taking the most credible N positive/negative interactions

In [31]:
# N_train = 1500
# N_val = 500
# N_test = 500

In [32]:
# pos_train, neg_train = pos_P0[:N_train//2], neg_P0[:N_train//2]
# pos_val, neg_val = pos_P1[:N_val//2], neg_P1[:N_val//2]
# pos_test, neg_test = pos_P2[:N_test//2], neg_P2[:N_test//2]
# # Tomamos las siguientes 500 instancias dentro del conjunto P2 para generar un segundo test set
# pos_test2, neg_test2 = pos_P2[N_test//2:N_test], neg_P2[N_test//2:N_test]

In [33]:
# # Si queremos garantizar que no se repitan proteínas entre los conjuntos de test
# used_prots = np.unique(
#     np.concatenate([
#         pos_test[["Head", "Tail"]].values.flatten(),
#         neg_test[["Head", "Tail"]].values.flatten()
#     ])
# )

# # Candidatos para test2: interacciones de P2 que no usan ninguna proteína de used_prots
# mask_pos2 = ~(
#     pos_P2["Head"].isin(used_prots) |
#     pos_P2["Tail"].isin(used_prots)
# )
# mask_neg2 = ~(
#     neg_P2["Head"].isin(used_prots) |
#     neg_P2["Tail"].isin(used_prots)
# )

# pos_candidates2 = pos_P2[mask_pos2]
# neg_candidates2 = neg_P2[mask_neg2]

# # Tomar los primeras N_test/2 de esos candidatos
# pos_test2, neg_test2 = pos_candidates2.iloc[:N_test//2], neg_candidates2.iloc[:N_test//2]


In [34]:
# plot_degrees(pos_train, neg_train, xlim=(0,30))

In [35]:
# plot_degrees(pos_val, neg_val, xlim=(0,30))

In [36]:
# plot_degrees(pos_test, neg_test, xlim=(0,30))

In [37]:
# plot_degrees(pos_test2, neg_test2, xlim=(0,30))

When sampling the most confident negative samples, they follow a something similar to a power law distribution, similar to that of the positives. Thus learning by node degree should be impossible.

Save the interactions we are doing in case we want to augment the dataset in the future:

In [38]:
# pos_train.to_csv("/home/jovyan/TFG/DatosModelos/pos_train.csv")
# neg_train.to_csv("/home/jovyan/TFG/DatosModelos/neg_train.csv")
# pos_val.to_csv("/home/jovyan/TFG/DatosModelos/pos_val.csv")
# neg_val.to_csv("/home/jovyan/TFG/DatosModelos/neg_val.csv")
# pos_test.to_csv("/home/jovyan/TFG/DatosModelos/pos_test.csv")
# neg_test.to_csv("/home/jovyan/TFG/DatosModelos/neg_test.csv")
# pos_test2.to_csv("/home/jovyan/TFG/DatosModelos/RepCientifica_pos_test.csv")
# neg_test2.to_csv("/home/jovyan/TFG/DatosModelos/RepCientifica_neg_test.csv")

In [39]:
pos_train = pd.read_csv("/home/jovyan/TFG/DatosModelos/pos_train.csv")
neg_train = pd.read_csv("/home/jovyan/TFG/DatosModelos/neg_train.csv")
pos_val = pd.read_csv("/home/jovyan/TFG/DatosModelos/pos_val.csv")
neg_val = pd.read_csv("/home/jovyan/TFG/DatosModelos/neg_val.csv")
pos_test = pd.read_csv("/home/jovyan/TFG/DatosModelos/pos_test.csv")
neg_test = pd.read_csv("/home/jovyan/TFG/DatosModelos/neg_test.csv")
pos_test2 = pd.read_csv("/home/jovyan/TFG/DatosModelos/RepCientifica_pos_test.csv")
neg_test2 = pd.read_csv("/home/jovyan/TFG/DatosModelos/RepCientifica_neg_test.csv")

In [40]:
train_prots = np.unique(np.concatenate([df[["Head", "Tail"]].values.flatten() for df in (pos_train, neg_train)]))
val_prots = np.unique(np.concatenate([df[["Head", "Tail"]].values.flatten() for df in (pos_val, neg_val)]))
test_prots = np.unique(np.concatenate([df[["Head", "Tail"]].values.flatten() for df in (pos_test, neg_test)]))
test_prots2 = np.unique(np.concatenate([df[["Head", "Tail"]].values.flatten() for df in (pos_test2, neg_test2)]))

Make sure the mice proteins used to predict interactions with effectors were not present in the different training sets

In [41]:
# np.intersect1d(train_prots, val_prots).size 
# np.intersect1d(train_prots, test_prots).size 
# np.intersect1d(val_prots, test_prots).size
# np.intersect1d(test_prots, test_prots2).size
# # np.intersect1d(pos_test, pos_test2).size
np.intersect1d(train_prots, uniq_prots).size
np.intersect1d(val_prots, uniq_prots).size
np.intersect1d(test_prots, uniq_prots).size

6

We want to know which proteins are repeated in the datasets

In [42]:
common_prots = np.intersect1d(train_prots, uniq_prots)
print(len(common_prots))   
print(common_prots)        

28
['O88351' 'O88522' 'P06804' 'P11672' 'P25118' 'P39429' 'P62908' 'P70196'
 'Q3ULB5' 'Q3UP24' 'Q60680' 'Q60803' 'Q60855' 'Q61160' 'Q62159' 'Q6NZP1'
 'Q8C0G2' 'Q8CF89' 'Q91V41' 'Q99K90' 'Q9D1G1' 'Q9D2Y4' 'Q9D620' 'Q9D7G9'
 'Q9EPB4' 'Q9EPQ1' 'Q9QUK6' 'Q9QY24']


In [43]:
common_prots = np.intersect1d(val_prots, uniq_prots)
print(len(common_prots))   
print(common_prots)  

8
['O88336' 'O88643' 'P18340' 'P63085' 'Q61179' 'Q8C015' 'Q8R361' 'Q9JIL2']


In [44]:
common_prots = np.intersect1d(test_prots, uniq_prots)
print(len(common_prots))   
print(common_prots)  

6
['P47811' 'P49138' 'P61161' 'Q8BH93' 'Q9JIB6' 'Q9WUI1']


We are going to change the interactions csv to randomize the order in which the effectors are listed in case of a tie between shortest path and shared connections between two pairs of effector-protein pairs

In [47]:
# Add a column with random numbers
effector_protein_df['Random_Tiebreaker'] = np.random.rand(len(effector_protein_df))

# Stablish the order criteria
eff_prot_df_ordered = effector_protein_df.sort_values(
    by=['Shortest_Path', 'Shared_Connections', 'Random_Tiebreaker'],
    ascending=[True, False, True] 
)

# We take out the last column as we do not need it anymore
eff_prot_df_ordered = eff_prot_df_ordered.drop(columns=['Random_Tiebreaker'])

# Store the result in a file
eff_prot_df_ordered.to_csv("/home/jovyan/TFG/Interacciones_EffectorProteina_LiteratureOnly_Random_Ordenadas.csv", index=False)

## Write json input files for AlphaFold3

In [46]:
# # Reload sequences
# uniq_prots = np.unique(np.concatenate([df[["Head", "Tail"]].values.flatten() for df in (pos_train, neg_train, pos_val, neg_val, pos_test, neg_test, pos_test2, neg_test2)]))

# seq_dict = {}

# for protein in uniq_prots:
#     try:
#         records = list(SeqIO.parse(f"Secuencias_ProteinaProteina/{protein}.fasta", format="fasta"))
#         if len(records) > 1:
#             print(f"{protein} has more than one sequence!")
#         seq_dict[protein] = str(records[0].seq)

#     except Exception as e:
#         print(f"Fail at {protein}")
#         print(records)
#         raise e

In [47]:
import json
def to_input(name, seq1, seq2):
    sequences = [
        {"protein": {
            "id": "A",
            "sequence": seq1
            }
        },
        {"protein": {
            "id": "B",
            "sequence": seq2
            }
        },
    ]

    alphafold_json = json.dumps(
        {
            'dialect': "alphafold3",
            'version': 1,
            'name': name,
            'sequences': sequences,
            'modelSeeds': [0],
        },
        indent=2,
    )
    return alphafold_json

def write_block_inputs(effector_protein_df):
    for idx, row in effector_protein_df.iterrows():
        id1 = row["Protein"]
        id2 = row["Effector"]
        seq_prot = prots_seq_dict[id1]
        seq_eff = eff_seq_dict[id2]

        name = f"{id1}_{id2}"

        jsoncontent = to_input(name, seq_prot, seq_eff)
        with open(INPUTS_FOLDER / (name + ".json"), "w") as file:
            file.write(jsoncontent)

# def write_block_inputs(pos_df, neg_df, block, folder, seq_dict):
#     for idx, row in pos_df.iterrows():
#         id1 = row["Head"]
#         id2 = row["Tail"]
#         seq1 = seq_dict[id1]
#         seq2 = seq_dict[id2]

#         name = f"{block}_{id1}_{id2}_1"

#         jsoncontent = to_input(name, seq1, seq2)
#         with open(INPUTS_FOLDER / (name + ".json"), "w") as file:
#             file.write(jsoncontent)

#     for idx, row in neg_df.iterrows():
#         id1 = row["Head"]
#         id2 = row["Tail"]
#         seq1 = seq_dict[id1]
#         seq2 = seq_dict[id2]

#         name = f"{block}_{id1}_{id2}_0"

#         jsoncontent = to_input(name, seq1, seq2)
#         with open(INPUTS_FOLDER / (name + ".json"), "w") as file:
#             file.write(jsoncontent)

In [48]:
INPUTS_FOLDER = Path("./inputs_Efectores_mmseqs")
write_block_inputs(effector_protein_df)
# write_block_inputs(pos_train, neg_train, "P0", INPUTS_FOLDER, seq_dict)
# write_block_inputs(pos_val, neg_val, "P1", INPUTS_FOLDER, seq_dict)
# write_block_inputs(pos_test, neg_test, "P2", INPUTS_FOLDER, seq_dict)
#write_block_inputs(pos_test2, neg_test2, "P2", INPUTS_FOLDER, seq_dict)

In [49]:
len(list(INPUTS_FOLDER.glob("*.json")))

5472

In [50]:
# train_prots = np.unique(np.array([filename.name.split("_")[1:3] for filename in INPUTS_FOLDER.glob("P0*")]).flatten())
# val_prots = np.unique(np.array([filename.name.split("_")[1:3] for filename in INPUTS_FOLDER.glob("P1*")]).flatten())
# test_prots = np.unique(np.array([filename.name.split("_")[1:3] for filename in INPUTS_FOLDER.glob("P2*")]).flatten())
# test_prots2 = np.unique(np.array([filename.name.split("_")[1:3] for filename in INPUTS_FOLDER.glob("P2*")]).flatten())

In [51]:
# np.intersect1d(train_prots, val_prots)

In [52]:
# np.intersect1d(train_prots, test_prots)

In [53]:
# np.intersect1d(val_prots, test_prots)

In [54]:
# np.intersect1d(train_prots, val_prots)

In [55]:
# np.intersect1d(test_prots, test_prots2)

### Effector json files

Vamos a repetir la creación de los archivos json para mmseqs pero solo para los efectores, así podremos ver si los embeddings de los efectores siguen una cierta agrupación que nos pueda servir para agruparlos y evitar el data leakage cuando hagamos la validación cruzada.

In [60]:
import json
def to_input(name, seq1):
    sequences = [
        {"protein": {
            "id": "A",
            "sequence": seq1
            }
        }
    ]

    alphafold_json = json.dumps(
        {
            'dialect': "alphafold3",
            'version': 1,
            'name': name,
            'sequences': sequences,
            'modelSeeds': [0],
        },
        indent=2,
    )
    return alphafold_json

def write_block_inputs(effector_df):
    for idx, row in effector_protein_df.iterrows():
        id1 = row["Effector"]
        seq_eff = eff_seq_dict[id1]

        name = id1

        jsoncontent = to_input(name, seq_eff)
        with open(INPUTS_FOLDER / (name + ".json"), "w") as file:
            file.write(jsoncontent)

In [61]:
INPUTS_FOLDER = Path("./input_solo_Efectores_mmseqs")
write_block_inputs(effector_protein_df)

### Protein json files

Repetimos el mismo proceso de antes solo para proteínas.

In [19]:
import json
def to_input(name, seq1):
    sequences = [
        {"protein": {
            "id": "A",
            "sequence": seq1
            }
        }
    ]

    alphafold_json = json.dumps(
        {
            'dialect': "alphafold3",
            'version': 1,
            'name': name,
            'sequences': sequences,
            'modelSeeds': [0],
        },
        indent=2,
    )
    return alphafold_json

def write_block_inputs(protein_df):
    for idx, row in effector_protein_df.iterrows():
        id1 = row["Protein"]
        seq_eff = prots_seq_dict[id1]

        name = id1

        jsoncontent = to_input(name, seq_eff)
        with open(INPUTS_FOLDER / (name + ".json"), "w") as file:
            file.write(jsoncontent)

In [20]:
INPUTS_FOLDER = Path("./input_solo_Proteinas_mmseqs")
write_block_inputs(effector_protein_df)